# Constraint Analysis (preliminary)
### Roadrunner propulsion reverse-engineering study · notebook 6 of 14

**Purpose.** Turn the performance requirements (an input, ADR-0027) into curves of installed
thrust loading $T_{SL}/W_{TO}$ against wing loading, in the manner of UCK 474E Lecture 2, and
read off the thrust the engine pair must give. It answers:

> *Which requirement sizes the engine, and how much room do the others leave?*

**This is a first iteration and rough by intent (ADR-0029).** The engine is an algebraic
turbojet thrust lapse and the drag polar is the owner's hand estimate (tier C, ±30 %). The
cycle analysis closes the numbers later and replaces the lapse. Read every number here as
*consistent with the assumptions*, not as a property of the vehicle.

Mission and constraint are coupled: a requirement is evaluated at the weight fraction $\beta$ at
the start of its mission segment, and the mission fuel depends on the thrust the constraints
demand. `solve_design_point()` iterates the two to a fixed point; the mission side is the next
notebook.

This is the *preliminary* analysis (ADR-0031). It is repeated in notebook 10 with the engine's own
thrust lapse and part-throttle fuel consumption from the off-design cycle analysis; compare the two.

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from roadrunner_re import analysis_outputs as ao
from roadrunner_re import reports
from roadrunner_re.analysis_inputs import load_analysis_inputs
from roadrunner_re.design_point import solve_design_point
from roadrunner_re.notebook import nb_setup
from roadrunner_re.plots import analysis as plots

paths = nb_setup()
inputs = load_analysis_inputs()
design = solve_design_point(inputs)          # the single call site (ADR-0029)
sweep = ao.takeoff_mass_sweep(inputs, ao.sweep_masses(inputs))


---
## 1 · The engine model

Installed thrust at maximum power as a fraction of installed sea-level static thrust, for a
turbojet (Lecture 2, after Mattingly, Heiser and Pratt), with $\theta_0$ and $\delta_0$ the
total-to-standard temperature and pressure ratios on the mission's hot day:

$$\alpha = \delta_0\left\{1 - 0.3(\theta_0-1) - 0.1\sqrt{M_0}\;\left[-\,1.5\,\frac{\theta_0-TR}{\theta_0}\ \text{if}\ \theta_0>TR\right]\right\}$$

The throttle ratio $TR$ is a cycle choice not yet made; it is set to 1.00. The coefficients
are in `config/analysis.yaml` with their source.

In [ ]:
print(reports.lapse_table(inputs, [("launch, sea level static", 0, 0.0), ("climb, sea level", 0, 0.6),
                                   ("dash", 3000, 0.8), ("loiter", 3000, 0.5), ("return", 3000, 0.6)]))

---
## 2 · The requirements as curves

With $x = W_{TO}\,g/S_{ref}$, and $S_{ref}$ the body cross-section of the hand estimate (so
this is *not* wing loading in the usual sense),

$$\frac{T_{SL}}{W_{TO}} = \frac{\beta}{\alpha}\left[K_1 n^2\frac{\beta}{q}x + K_2 n + \frac{C_{D0}+C_{DR}}{(\beta/q)\,x} + \frac{P_s}{V}\right]$$

Level flight ($P_s=0$), climb ($P_s$ = rate of climb), sustained turn ($n>1$) and ceiling
(small rate of climb) are cases I, II, III and VIII of the lecture. Vertical launch requires
$T/W = $ margin$/\alpha$ at sea-level static on the hot day. The landing floor is an *upper*
bound: idle thrust must not exceed a fraction of landing weight.

In [ ]:
print(reports.design_constraint_table(design))
print(f"\nconverged in {design.iterations} iterations")

---
## 3 · The constraint diagram

In [ ]:
fig = plots.constraint_diagram(inputs, design, paths.figures / "constraint_diagram.png")

The shaded region is where every lower bound is met. The landing floor lies far above the plotted
range (idle thrust is a small fraction of take-off thrust), so it does not appear. The dotted
line is the nominal take-off mass; the top axis converts wing loading to take-off mass.

---
## 4 · Take-off mass sweep

The wing loading moves with take-off mass because the reference area is fixed. The take-off mass
itself is uncertain (one secondary source, 40 to 150 kg bounds), so the thrust requirement is
swept over that range.

In [ ]:
fig = plots.thrust_sweep_figure(sweep, paths.figures / "thrust_sweep.png")

---
## 5 · Handoff

In [ ]:
handoff = ao.write_handoff(ao.constraints_out(inputs, design, sweep), paths.out / "constraints_preliminary.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- **Vertical launch sizes the engine**, as ADR-0009 expected: the launch margin over weight on a
  hot day sets the thrust loading; dash, climb, manoeuvre and loiter ceiling all fall below it
  at the nominal take-off mass.
- **At low take-off mass the dash binds instead**, because the wing loading falls and induced
  drag grows. The crossover is visible in the sweep.
- **The landing floor leaves a very wide window** with the catalogue idle fraction. It would
  matter only for an engine that cannot throttle far down.
- **Standing finding:** the polar is constant with Mach number (no drag rise), the lapse is a
  generic algebraic model, installation losses are not applied and the launch margin is an
  assumption. The thrust loading is therefore indicative; the cycle analysis closes it.